# Day 2 — Directed-flow QUBO and Ising Hamiltonian

This notebook is a short teaching view of reusable logic in `src/`. It derives and exhaustively validates the mathematical cost model before any QAOA circuit is constructed.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
from IPython.display import Image, display
from exact_reference import compute_exact_reference
from graph import edge_order_records, load_graph
from ising import max_qubo_ising_error, qubo_to_ising
from qubo import (build_qubo, derive_critical_penalty, enumerate_state_space,
                  flow_equation_strings, flow_feasibility_verdict,
                  flow_penalty_coefficients, frozen_penalty_grid,
                  max_qubo_expansion_error)

## 1. Load the immutable Day-1 graph

The edge variables retain the frozen lexicographic `q0 → q13` order.

In [ ]:
graph_path = PROJECT_ROOT / 'data' / 'graph.json'
graph = load_graph(graph_path)
reference, day1_routes = compute_exact_reference(graph, graph_path=graph_path)
display(pd.DataFrame(edge_order_records(graph)))
reference['exact_reference']

## 2. Directed flow equations

For each node, `f_v(x) = outgoing − incoming − b_v`; the penalty is `P_flow(x)=Σ_v f_v(x)²`.

In [ ]:
print('\n'.join(flow_equation_strings(graph)))
flow_qubo = flow_penalty_coefficients(graph)
print(f'Expanded P_flow: constant={flow_qubo.constant}, '
      f'{len(flow_qubo.linear)} linear and {len(flow_qubo.pair)} nonzero pair coefficients')

## 3. Exhaust all 2¹⁴ edge selections

The independent path decoder and zero flow penalty must select exactly the same states.

In [ ]:
states = enumerate_state_space(graph)
verdict = flow_feasibility_verdict(states)
assert verdict['sets_identical']
assert verdict['decoder_valid_route_count'] == len(day1_routes)
verdict

## 4. Derive the classical penalty threshold and freeze the grid

`A_crit` is computed from every cheaper infeasible state, before any quantum result exists.

In [ ]:
C_star = reference['exact_reference']['cost']
A_crit, critical_states = derive_critical_penalty(states, C_star)
penalty_grid = frozen_penalty_grid(A_crit)
print('A_crit =', A_crit)
print('critical state(s):', [state.canonical_bitstring for state in critical_states])
pd.DataFrame([{'label': choice.label, 'A': float(choice.value)} for choice in penalty_grid])

## 5. Canonical QUBO → Ising mapping

Using `x_i=(I−Z_i)/2`, compare all 16,384 basis energies for all four frozen penalties, including the identity constant.

In [ ]:
penalties = tuple(choice.value for choice in penalty_grid)
qubos = tuple(build_qubo(graph, A) for A in penalties)
qubo_error = max_qubo_expansion_error(graph, states, penalties)
ising_error = max_qubo_ising_error(states, qubos)
assert qubo_error == 0 and ising_error == 0
{'states': len(states), 'QUBO max error': float(qubo_error),
 'Ising max error': float(ising_error),
 'example Ising constant at A=2': float(qubo_to_ising(qubos[0]).constant)}

## 6. Presentation figures

In [ ]:
display(Image(filename=PROJECT_ROOT / 'figures' / '03_graph_qubo_ising_structure.png', width=1000))
display(Image(filename=PROJECT_ROOT / 'figures' / '04_penalty_state_space_geometry.png', width=1000))

In [ ]:
print(f'Day 2 summary: {len(states)} states; {verdict["flow_penalty_zero_count"]} valid routes; '
      f'A_crit={A_crit}; grid={[int(choice.value) for choice in penalty_grid]}; '
      f'QUBO/Ising max error={float(ising_error)}')